# Ours vs Palash's T7 judge, on our own dumps

Question: does the T7 candidate judge (no agreement) and its early exit improve on our current best (agreement + escalation at bar 11, `agreement_query_pipeline.py`),
and should parts of the old rule be removed when it goes in?

Data: rich dumps made on this machine by `Rich_Dump_Run.ipynb` (all 27 queries, raw LoFTR matches). `official150`: 150 random official photos (the judge was trained on the
official set, so this one is **in-sample for the judge**). `nudt707`: 150 NUDT photos that no T6/T7 set used (**clean test**). Judge and rule are Palash's frozen ones, nothing is refitted.
Scores are out of 60; wrong = beyond 8 sigma (-20 each). Everything is a replay, no GPU.

In [1]:
import os, sys, json
os.environ.setdefault("OMP_NUM_THREADS", "6")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import joblib, numpy as np, pandas as pd
from train.checks import replay, query_policy
from train.checks.decision_signals import build_signal_table
from train.checks.candidate_judge import add_context, add_features, baseline, decide_frozen, judge_new, summary, paired_gain
T7 = ROOT / "tests/Test_Iteration_7/results"
DUMPS = T7 / "rich_dump"
SETS = {"official150": "official150", "nudt707": "nudt_fresh707"}
frozen = json.load(open(T7 / "frozen_rules_t7.json")); model = joblib.load(T7 / "judge_t7.joblib")
photos = {s: replay.load_set(DUMPS / f) for s, f in SETS.items()}
print({s: len(p) for s, p in photos.items()})

{'official150': 150, 'nudt707': 150}


## 1. Our current best, replayed from the dump (and with the DINO-z veto of T6)

In [2]:
rows = []
for s, ph in photos.items():
    for pid, photo in ph.items():
        r = replay.decide_current(photo); rows.append(dict(data=s, photo=pid, stage=r["stage"], pts=int(r["points"]), err_sigma=r["err_sigma"], n_queries=r["n_queries"], inliers=r["inliers"]))
        v = query_policy.run_policy(photo, z_veto=(1.3, 15))
        rows.append(dict(data=s + "|veto", photo=pid, stage=v["stage"], pts=int(v["points"]) if v["answered"] else 0, err_sigma=v["err_sigma"], n_queries=v["n_queries"], inliers=np.nan))
ours_all = pd.DataFrame(rows)
ours = ours_all[~ours_all.data.str.contains("veto")]
veto = ours_all[ours_all.data.str.contains("veto")].assign(data=lambda d: d.data.str.replace("|veto", "", regex=False))
print(summary(ours)); print(summary(veto))

             score  wrong  answered  queries    n  n_answered
data                                                         
official150  52.80      0     0.893     8.38  150         134
nudt707      53.84      1     0.920     7.82  150         138
             score  wrong  answered  queries    n  n_answered
data                                                         
official150  52.80      0     0.893    8.247  150         134
nudt707      53.44      1     0.913    7.847  150         137


## 2. Signals after fine matching, then the frozen judge

In [3]:
tabs = {}
for s, f in SETS.items():
    t = build_signal_table(DUMPS / f, s, 10)
    t.to_csv(T7 / f"signals_ours_{s}.csv.gz", index=False)
    tabs[s] = add_context(add_features(t)).reset_index(drop=True)
table = pd.concat(tabs.values(), ignore_index=True)
p = judge_new(model, table)
print({s: (len(t), t.photo.nunique()) for s, t in tabs.items()})

{'official150': (40500, 150), 'nudt707': (40500, 150)}


In [4]:
res = {}
for variant, cfg in frozen["variants"].items():
    res[variant] = decide_frozen(table, p, cfg["tau"], variant)
base_pc = baseline(table, 8, 11, min_choice=3)          # bars only, no agreement
base_jet = baseline(table, 8, 8, jetson=True, m=5)      # today's Jetson rule
out = {"ours (agree+escal)": ours, "ours + z-veto": veto, "bars only (no agree)": base_pc, "judge pc": res["pc"], "judge pc + early exit": res["pc_early"],
       "bars only, Jetson": base_jet, "judge Jetson": res["jetson"], "judge Jetson + early exit": res["jetson_early"]}
tab = pd.concat([summary(v).assign(rule=k).reset_index() for k, v in out.items()])
tab = tab[["rule", "data", "score", "wrong", "answered", "queries", "n"]]
tab

,rule,data,score,wrong,answered,queries,n
0,ours (agree+escal),official150,52.80,0,0.893,8.380,150
1,ours (agree+escal),nudt707,53.84,1,0.920,7.820,150
0,ours + z-veto,official150,52.80,0,0.893,8.247,150
1,ours + z-veto,nudt707,53.44,1,0.913,7.847,150
0,bars only (no agree),official150,50.00,0,0.847,9.360,150
1,bars only (no agree),nudt707,53.04,1,0.907,8.100,150
0,judge pc,official150,56.80,0,0.960,7.260,150
1,judge pc,nudt707,53.44,1,0.913,8.100,150
0,judge pc + early exit,official150,55.40,0,0.960,3.200,150
1,judge pc + early exit,nudt707,53.28,1,0.913,3.753,150


## 3. Paired gain over our current best (95 % bootstrap), PC rules

In [5]:
rows = []
for k in ["bars only (no agree)", "ours + z-veto", "judge pc", "judge pc + early exit"]:
    g = paired_gain(ours, out[k], list(SETS)); rows.append(dict(rule=k, **g))
pd.DataFrame(rows)

,rule,official150,nudt707
0,bars only (no agree),"-2.80 [-4.81, -0.80]","-0.80 [-2.00, +0.00]"
1,ours + z-veto,"+0.00 [+0.00, +0.00]","-0.40 [-1.20, +0.00]"
2,judge pc,"+4.00 [+1.60, +6.80]","-0.40 [-1.60, +0.80]"
3,judge pc + early exit,"+2.60 [-0.16, +5.56]","-0.56 [-2.00, +0.80]"


## 4. Hybrids: keep our rule and add the judge as a second key

`AND`: answer only if both answer (judge as veto). `OR`: our answer, else the judge's (judge as rescue). Points come from the rule that supplies the place.

In [6]:
def merge(a, b, mode):
    j = a.set_index(["data", "photo"]).join(b.set_index(["data", "photo"]), rsuffix="_j")
    A, B = j.stage != "skip", j.stage_j != "skip"
    if mode == "and":
        pts = np.where(A & B, j.pts, 0); st = np.where(A & B, j.stage, "skip"); q = j.n_queries
    else:
        pts = np.where(A, j.pts, np.where(B, j.pts_j, 0)); st = np.where(A, j.stage, np.where(B, j.stage_j, "skip")); q = np.where(A, j.n_queries, np.maximum(j.n_queries, j.n_queries_j))
    return pd.DataFrame(dict(pts=pts, stage=st, n_queries=q), index=j.index).reset_index()
hy = {"ours AND judge pc": merge(ours, res["pc"], "and"), "ours OR judge pc": merge(ours, res["pc"], "or"),
      "ours+veto AND judge pc": merge(veto, res["pc"], "and"), "ours+veto OR judge pc": merge(veto, res["pc"], "or")}
tab2 = pd.concat([summary(v).assign(rule=k).reset_index() for k, v in hy.items()])[["rule", "data", "score", "wrong", "answered", "queries", "n"]]
tab2

,rule,data,score,wrong,answered,queries,n
0,ours AND judge pc,official150,52.40,0,0.887,8.380,150
1,ours AND judge pc,nudt707,53.04,1,0.907,7.820,150
0,ours OR judge pc,official150,57.20,0,0.967,8.380,150
1,ours OR judge pc,nudt707,54.24,1,0.927,7.820,150
0,ours+veto AND judge pc,official150,52.40,0,0.887,8.247,150
1,ours+veto AND judge pc,nudt707,53.04,1,0.907,7.847,150
0,ours+veto OR judge pc,official150,57.20,0,0.967,8.247,150
1,ours+veto OR judge pc,nudt707,53.84,1,0.920,7.847,150


## 5. Photos where the judge and our rule disagree (pc)

In [7]:
j = ours.set_index(["data", "photo"]).join(res["pc"].set_index(["data", "photo"]), rsuffix="_judge")
d = j[j.pts != j.pts_judge][["stage", "inliers", "pts", "stage_judge", "inliers_judge", "p", "pts_judge", "err_sigma_judge"]]
d.round(2)

stage  inliers  pts stage_judge  inliers_judge     p  \
data        photo                                                             
official150 7           skip      5.0    0        std6            4.0  0.95   
            36     agreement      3.0   10        skip            5.0  0.23   
            66          skip     10.0    0        wide           10.0  0.99   
            172         skip      9.0    0        std6            4.0  0.97   
            176         skip      5.0    0        std6            4.0  0.95   
            200         skip      4.0    0        std6            3.0  0.93   
            217         skip      4.0    0        std6            3.0  0.96   
            266         skip      4.0    0        std6            4.0  0.92   
            297         skip      6.0    0        std6            4.0  0.99   
            329         skip      7.0    0        std6            7.0  0.99   
            338         skip      6.0    0        std6            6.0  0.99   
            376         skip      5.0    0        std6            5.0  0.99   
nudt707     1636        skip      8.0    0        wide            8.0  1.00   
            2066   agreement      6.0   10        skip            7.0  0.59   
            2181   agreement      6.0   10        skip            4.0  0.36   

                   pts_judge  err_sigma_judge  
data        photo                              
official150 7             10             0.11  
            36             0             0.53  
            66            10             0.55  
            172           10             0.39  
            176           10             0.22  
            200           10             0.72  
            217           10             0.73  
            266           10             0.58  
            297           10             0.48  
            329           10             0.31  
            338           10             0.31  
            376           10             0.56  
nudt707     1636          10             0.92  
            2066           0             0.10  
            2181           0           156.52

In [8]:
tab.to_csv(T7 / "ours_check_summary.csv", index=False); tab2.to_csv(T7 / "ours_check_hybrids.csv", index=False); d.to_csv(T7 / "ours_check_disagreements.csv")

## 6. Early exit on top of our rule

Run the 6 standard queries one at a time; when the judge's P for the current choice is >= 0.98, answer at once (stage `early`). If it never fires, our rule runs unchanged (bar 8 / agreement, then 27 queries at bar 11).
Replay: take the judge's `early` answers from the judge + early exit run, everything else from ours.

In [9]:
e = res["pc_early"].set_index(["data", "photo"]); o = ours.set_index(["data", "photo"])
fired = e.stage == "early"
mix = o.copy()
for col in ["stage", "pts", "n_queries", "err_sigma"]:
    mix.loc[fired[fired].index, col] = e.loc[fired[fired].index, col]
mix = mix.reset_index()
tab3 = pd.concat([summary(mix).assign(rule="ours + judge early exit").reset_index(), summary(ours).assign(rule="ours").reset_index()])[["rule", "data", "score", "wrong", "answered", "queries", "n"]]
print(fired.groupby(level=0).sum().to_dict(), "early exits")
print(paired_gain(ours, mix, list(SETS)))
tab3

{'nudt707': 134, 'official150': 132} early exits
{'official150': '+0.20 [-1.64, +2.20]', 'nudt707': '-0.16 [-0.48, +0.00]'}


,rule,data,score,wrong,answered,queries,n
0,ours + judge early exit,official150,53.00,0,0.920,3.760,150
1,ours + judge early exit,nudt707,53.68,1,0.920,3.473,150
0,ours,official150,52.80,0,0.893,8.380,150
1,ours,nudt707,53.84,1,0.920,7.820,150


In [10]:
tab3.to_csv(T7 / "ours_check_early_exit.csv", index=False)